In [6]:
import pandas as pd

In [7]:
df = pd.read_csv("household_power.csv", delimiter=";")
df.head()

/tmp/ipykernel_18777/1223318118.py:1: DtypeWarning: Columns (0: Global_active_power, 1: Global_reactive_power, 2: Voltage, 3: Global_intensity, 4: Sub_metering_1, 5: Sub_metering_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("household_power.csv", delimiter=";")


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [8]:
# df.drop(columns = ['Date', 'Time']).corr()

# Fix missing values

In [9]:
df.tail()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
2075254,26/11/2010,20:58:00,0.946,0.0,240.43,4.0,0.0,0.0,0.0
2075255,26/11/2010,20:59:00,0.944,0.0,240.0,4.0,0.0,0.0,0.0
2075256,26/11/2010,21:00:00,0.938,0.0,239.82,3.8,0.0,0.0,0.0
2075257,26/11/2010,21:01:00,0.934,0.0,239.7,3.8,0.0,0.0,0.0
2075258,26/11/2010,21:02:00,0.932,0.0,239.55,3.8,0.0,0.0,0.0


In [10]:
df.shape

(2075259, 9)

In [11]:
df["Datetime"] = pd.to_datetime(df['Date'] + " " + df["Time"], dayfirst = True)
df = df.set_index("Datetime")

In [12]:
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,,,
2006-12-16 17:24:00,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
2006-12-16 17:25:00,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2006-12-16 17:26:00,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
2006-12-16 17:27:00,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
2006-12-16 17:28:00,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [13]:
df = df.sort_index()
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,,,
2006-12-16 17:24:00,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
2006-12-16 17:25:00,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2006-12-16 17:26:00,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
2006-12-16 17:27:00,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
2006-12-16 17:28:00,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [15]:
full_index = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="min"
)

missing = full_index.difference(df.index)

print("Missing timestamps:", len(missing))

Missing timestamps: 0


In [16]:
gaps = df.index.to_series().diff()

# show rows where gap > 1 minute
gap_rows = gaps[gaps > pd.Timedelta(minutes=1)]

gap_rows.head()

Series([], Name: Datetime, dtype: timedelta64[us])

In [17]:
df.isna().sum()

Date                         0
Time                         0
Global_active_power          0
Global_reactive_power        0
Voltage                      0
Global_intensity             0
Sub_metering_1               0
Sub_metering_2               0
Sub_metering_3           25979
dtype: int64

In [18]:
df["Sub_metering_3"] = df["Sub_metering_3"].ffill().bfill()
df.isna().sum()

Date                     0
Time                     0
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

In [19]:
df.dtypes

Date                         str
Time                         str
Global_active_power       object
Global_reactive_power     object
Voltage                   object
Global_intensity          object
Sub_metering_1            object
Sub_metering_2            object
Sub_metering_3           float64
dtype: object

In [20]:
cols = df.columns.difference(["Date", "Time"])
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")


# Fix anomolies

## Checking for spikes

In [21]:
mask = (df["Voltage"] == 0) & (df["Global_active_power"] != 0)

bad_rows = df[mask]

print("Count:", len(bad_rows))
bad_rows.head()


Count: 0


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,,,


In [22]:
df.dtypes

Date                         str
Time                         str
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

In [23]:
col = "Global_active_power"

mean = df[col].mean()
std = df[col].std()

z = (df[col] - mean) / std

spikes = df[abs(z) > 6]   # threshold = 6σ (very extreme)

print("Spike count:", len(spikes))
spikes.head()


Spike count: 1001


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,,,
2006-12-16 17:45:00,16/12/2006,17:45:00,7.706,0.000,230.98,33.2,0.0,0.0,17.0
2006-12-19 08:47:00,19/12/2006,08:47:00,7.828,0.182,232.20,33.6,36.0,72.0,17.0
2006-12-19 08:48:00,19/12/2006,08:48:00,7.840,0.188,232.55,33.6,36.0,71.0,16.0
2006-12-19 08:49:00,19/12/2006,08:49:00,7.818,0.202,232.22,33.6,37.0,71.0,17.0
2006-12-19 08:50:00,19/12/2006,08:50:00,7.816,0.184,232.39,33.6,36.0,71.0,16.0


In [24]:
q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1

lower = q1 - 5 * iqr
upper = q3 + 5 * iqr

spikes = df[(df[col] < lower) | (df[col] > upper)]

print("Spike count:", len(spikes))


Spike count: 790


In [25]:
delta = df["Global_active_power"].diff().abs()

jump_spikes = df[delta > delta.quantile(0.999)]

print("Jump spikes:", len(jump_spikes))
jump_spikes.head()


Jump spikes: 2049


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,,,
2006-12-16 18:35:00,16/12/2006,18:35:00,6.072,0.000,232.48,26.4,0.0,27.0,17.0
2006-12-16 23:50:00,16/12/2006,23:50:00,3.844,0.148,238.71,16.6,0.0,0.0,0.0
2006-12-17 01:06:00,17/12/2006,01:06:00,2.362,0.096,241.35,10.4,0.0,28.0,0.0
2006-12-17 01:49:00,17/12/2006,01:49:00,0.574,0.302,243.88,2.6,0.0,5.0,0.0
2006-12-17 01:54:00,17/12/2006,01:54:00,0.774,0.294,242.43,4.2,0.0,5.0,0.0


In [26]:
spikes = df[
    (df["Voltage"] < 150) |
    (df["Voltage"] > 300) |
    (df["Global_active_power"] < 0) |
    (df["Global_active_power"] > 20)
]

print("Physical-violation rows:", len(spikes))


Physical-violation rows: 0


In [27]:
z_mask = (df["Global_active_power"] - df["Global_active_power"].mean()).abs() > 6 * df["Global_active_power"].std()

q_low = df["Global_active_power"].quantile(0.0001)
q_high = df["Global_active_power"].quantile(0.9999)
q_mask = (df["Global_active_power"] < q_low) | (df["Global_active_power"] > q_high)

jump = df["Global_active_power"].diff().abs()
jump_mask = jump > jump.quantile(0.999)

combined_mask = z_mask | q_mask | jump_mask

print("Total flagged:", combined_mask.sum())


Total flagged: 3023


In [28]:
df.loc[combined_mask, "Global_active_power"] = pd.NA

df["Global_active_power"] = df["Global_active_power"].interpolate(method="time")


## Converting df from minutes to hourly

In [30]:
hourly = df.resample("h").agg({
    "Global_active_power": ["mean", "max", "min", "std"],
    "Voltage": ["mean", "max", "min", "std"],
    "Global_intensity": ["mean", "max", "min", "std"],
    "Sub_metering_1": ["mean", "max", "min", "std"],
    "Sub_metering_2": ["mean", "max", "min", "std"],
    "Sub_metering_3": ["mean", "max", "min", "std"],
})

hourly.head()

Global_active_power                             Voltage  \
                                   mean    max    min       std        mean   
Datetime                                                                      
2006-12-16 17:00:00            4.188278  7.026  2.472  1.070610  234.643889   
2006-12-16 18:00:00            3.598283  6.752  2.188  1.132717  234.580167   
2006-12-16 19:00:00            3.400233  4.070  3.202  0.135354  233.232500   
2006-12-16 20:00:00            3.268567  3.496  1.832  0.264286  234.071500   
2006-12-16 21:00:00            3.056467  3.502  1.692  0.576796  237.158667   

                                              Global_intensity        ...  \
                        max     min       std             mean   max  ...   
Datetime                                                              ...   
2006-12-16 17:00:00  237.14  230.98  1.528529        18.100000  33.2  ...   
2006-12-16 18:00:00  238.28  231.85  1.623576        15.600000  29.0  ...   
2006-12-16 19:00:00  235.12  231.57  0.901678        14.503333  17.4  ...   
2006-12-16 20:00:00  237.54  231.95  1.483866        13.916667  14.8  ...   
2006-12-16 21:00:00  239.69  235.73  0.862382        13.046667  14.8  ...   

                    Sub_metering_1      Sub_metering_2                        \
                               min  std           mean   max  min        std   
Datetime                                                                       
2006-12-16 17:00:00            0.0  0.0       0.527778   2.0  0.0   0.696362   
2006-12-16 18:00:00            0.0  0.0       6.716667  37.0  0.0  11.830858   
2006-12-16 19:00:00            0.0  0.0       1.433333   5.0  0.0   1.253695   
2006-12-16 20:00:00            0.0  0.0       0.000000   0.0  0.0   0.000000   
2006-12-16 21:00:00            0.0  0.0       0.416667   2.0  0.0   0.696034   

                    Sub_metering_3                        
                              mean   max   min       std  
Datetime                                                  
2006-12-16 17:00:00      16.861111  18.0  16.0  0.424451  
2006-12-16 18:00:00      16.866667  18.0  16.0  0.468198  
2006-12-16 19:00:00      16.683333  17.0  16.0  0.469102  
2006-12-16 20:00:00      16.783333  18.0  16.0  0.454420  
2006-12-16 21:00:00      17.216667  18.0  17.0  0.415450  

[5 rows x 24 columns]

In [31]:
hourly.columns = [
    f"{col[0]}_{col[1]}" for col in hourly.columns
]
hourly.head()

,Global_active_power_mean,Global_active_power_max,Global_active_power_min,Global_active_power_std,Voltage_mean,Voltage_max,Voltage_min,Voltage_std,Global_intensity_mean,Global_intensity_max,...,Sub_metering_1_min,Sub_metering_1_std,Sub_metering_2_mean,Sub_metering_2_max,Sub_metering_2_min,Sub_metering_2_std,Sub_metering_3_mean,Sub_metering_3_max,Sub_metering_3_min,Sub_metering_3_std
Datetime,,,,,,,,,,,,,,,,,,,,,
2006-12-16 17:00:00,4.188278,7.026,2.472,1.070610,234.643889,237.14,230.98,1.528529,18.100000,33.2,...,0.0,0.0,0.527778,2.0,0.0,0.696362,16.861111,18.0,16.0,0.424451
2006-12-16 18:00:00,3.598283,6.752,2.188,1.132717,234.580167,238.28,231.85,1.623576,15.600000,29.0,...,0.0,0.0,6.716667,37.0,0.0,11.830858,16.866667,18.0,16.0,0.468198
2006-12-16 19:00:00,3.400233,4.070,3.202,0.135354,233.232500,235.12,231.57,0.901678,14.503333,17.4,...,0.0,0.0,1.433333,5.0,0.0,1.253695,16.683333,17.0,16.0,0.469102
2006-12-16 20:00:00,3.268567,3.496,1.832,0.264286,234.071500,237.54,231.95,1.483866,13.916667,14.8,...,0.0,0.0,0.000000,0.0,0.0,0.000000,16.783333,18.0,16.0,0.454420
2006-12-16 21:00:00,3.056467,3.502,1.692,0.576796,237.158667,239.69,235.73,0.862382,13.046667,14.8,...,0.0,0.0,0.416667,2.0,0.0,0.696034,17.216667,18.0,17.0,0.415450


In [32]:
hourly.shape

(34589, 24)

In [33]:
hourly.isna().sum().sort_values(ascending=False).head()

Voltage_std             421
Voltage_min             421
Voltage_max             421
Voltage_mean            421
Global_intensity_std    421
dtype: int64

In [34]:
print(hourly[hourly.isna().any(axis=1)].shape)
hourly[hourly.isna().any(axis=1)].head()

(421, 24)


,Global_active_power_mean,Global_active_power_max,Global_active_power_min,Global_active_power_std,Voltage_mean,Voltage_max,Voltage_min,Voltage_std,Global_intensity_mean,Global_intensity_max,...,Sub_metering_1_min,Sub_metering_1_std,Sub_metering_2_mean,Sub_metering_2_max,Sub_metering_2_min,Sub_metering_2_std,Sub_metering_3_mean,Sub_metering_3_max,Sub_metering_3_min,Sub_metering_3_std
Datetime,,,,,,,,,,,,,,,,,,,,,
2007-04-28 01:00:00,0.487148,0.489207,0.485088,0.001219,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2007-04-28 02:00:00,0.482959,0.485018,0.480899,0.001219,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2007-04-28 03:00:00,0.478770,0.480829,0.476710,0.001219,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2007-04-28 04:00:00,0.474581,0.476640,0.472521,0.001219,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2007-04-28 05:00:00,0.470392,0.472451,0.468332,0.001219,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0


In [35]:
hourly = hourly.ffill().bfill()
hourly.isna().sum().sort_values(ascending=False).head()


Global_active_power_mean    0
Global_active_power_max     0
Global_active_power_min     0
Global_active_power_std     0
Voltage_mean                0
dtype: int64

In [36]:
hourly.to_csv("hourly_power_features.csv")